In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import logging

In [2]:
# Directories
csv_dir = "../CSV_Files"
Features_dir = "../Extracted_Features"

# Create necessary directories if they don't exist
os.makedirs("Combined_original_Frequency_Data", exist_ok=True)
os.makedirs("Biometrics_4HZ", exist_ok=True)
os.makedirs("Behavioral_4HZ", exist_ok=True)
os.makedirs("Facial_4HZ", exist_ok=True)

### ACC ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# Define column names for ACC data
columns = ['X', 'Y', 'Z']

acc_files = []

for root, dirs, files in os.walk(csv_dir):
    for filename in files:
        if "_ACC_D" in filename or "_ACC_A" in filename:
            acc_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if acc_files:
    for file_path in tqdm(acc_files, desc="Processing ACC files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            acc_data = pd.read_csv(file_path, header=None, names=columns)
            acc_data['user'] = user
            acc_data['session'] = session
            dataframes.append(acc_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_ACC = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} files.")

        # Save to CSV
        output_file = "Combined_original_Frequency_Data/combined_ACC_data_32HZ.parquet"
        combined_data_ACC.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")


#### Change Frequency ####

In [7]:
def change_frequency(df, current_hz, target_hz):

    # Function to downsample or upsample a group (user-session)
    def resample_group(group):
        # Time between samples in seconds
        time_per_sample_current = 1 / current_hz
        time_per_sample_target = 1 / target_hz
        
        if target_hz < current_hz:  # Downsampling
            # Resample by selecting every (current_hz / target_hz)th sample
            downsample_factor = int(current_hz / target_hz)
            return group.iloc[::downsample_factor, :].reset_index(drop=True)

        elif target_hz > current_hz:  # Upsampling
            # Resample by repeating rows to achieve the target Hz
            upsample_factor = int(target_hz / current_hz)
            return group.loc[group.index.repeat(upsample_factor)].reset_index(drop=True)

        else:
            # No change in frequency
            return group
    
    # Apply resampling for each (user, session) group with include_groups=False to suppress warning
    return df.groupby(['user', 'session'], group_keys=False).apply(resample_group)

In [ ]:
combined_data_ACC = change_frequency(combined_data_ACC, current_hz=32, target_hz=4)
combined_data_ACC.to_csv("Biometrics_4HZ/ACC_data_4HZ.csv", index=False)

### BVP ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# Define column names for BVP data
columns = ['BVP']

bvp_files = []

for root, dirs, files in os.walk(csv_dir):
    for filename in files:
        if "_BVP_D" in filename or "_BVP_A" in filename:
            bvp_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if bvp_files:
    for file_path in tqdm(bvp_files, desc="Processing BVP files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            bvp_data = pd.read_csv(file_path, header=None, names=columns)
            bvp_data['user'] = user
            bvp_data['session'] = session
            dataframes.append(bvp_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_BVP = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_BVP_data_64HZ.parquet"
        combined_data_BVP.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

### Change Frequency ###

In [ ]:
combined_data_BVP = change_frequency(combined_data_BVP, current_hz=64, target_hz=4)
combined_data_BVP.to_csv("Biometrics_4HZ/BVP_data_4HZ.csv", index=False)

### EDA ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# Define column names for EDA data
columns = ['EDA']

# List to store file paths
eda_files = []

# Walk through all subdirectories and collect matching files
for root, dirs, files in os.walk(csv_dir):
    for filename in files:
        if "_EDA_D" in filename or "_EDA_A" in filename:
            eda_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if eda_files:
    for file_path in tqdm(eda_files, desc="Processing EDA files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            eda_data = pd.read_csv(file_path, header=None, names=columns)
            eda_data['user'] = user
            eda_data['session'] = session
            dataframes.append(eda_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_EDA = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_EDA_data_4HZ.parquet"
        combined_data_EDA.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

In [10]:
combined_data_EDA.to_csv("Biometrics_4HZ/EDA_data_4HZ.csv", index=False)

### HR ####

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# Define column names for HR data
columns = ['HR']

# List to store file paths
hr_files = []

# Walk through all subdirectories and collect matching files
for root, dirs, files in os.walk(csv_dir):
    for filename in files:
        if "_HR_D" in filename or "_HR_A" in filename:
            hr_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if hr_files:
    for file_path in tqdm(hr_files, desc="Processing HR files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            hr_data = pd.read_csv(file_path, header=None, names=columns)
            hr_data['user'] = user
            hr_data['session'] = session
            dataframes.append(hr_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_HR = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_HR_data_1HZ.parquet"
        combined_data_HR.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

### Change Frequency ###

In [ ]:
combined_data_HR = change_frequency(combined_data_HR, current_hz=1, target_hz=4)
combined_data_HR.to_csv("Biometrics_4HZ/HR_data_4HZ.csv", index=False)

### TEMP ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# Define column names for TEMP data
columns = ['TEMP']

# List to store file paths
temp_files = []

# Walk through all subdirectories and collect matching files
for root, dirs, files in os.walk(csv_dir):
    for filename in files:
        if "_TEMP_D" in filename or "_TEMP_A" in filename:
            temp_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if temp_files:
    for file_path in tqdm(temp_files, desc="Processing TEMP files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            temp_data = pd.read_csv(file_path, header=None, names=columns)
            temp_data['user'] = user
            temp_data['session'] = session
            dataframes.append(temp_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_TEMP = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_TEMP_data_4HZ.parquet"
        combined_data_TEMP.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

In [14]:
combined_data_TEMP.to_csv("Biometrics_4HZ/TEMP_data_4HZ.csv", index=False)

### O2M (SPO2, Pulse_Rate, Motion) ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# Define column names for O2M data
columns = ['SPO2', 'pulse_rate', 'motion']

# List to store file paths
o2m_files = []

# Walk through all subdirectories and collect matching files
for root, dirs, files in os.walk(csv_dir):
    for filename in files:
        if "_O2M_D" in filename or "_O2M_A" in filename:
            o2m_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if o2m_files:
    for file_path in tqdm(o2m_files, desc="Processing O2M files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            o2m_data = pd.read_csv(file_path, header=None, names=columns)

            # Convert columns to numeric, forcing errors to NaN
            for col in columns:
                o2m_data[col] = pd.to_numeric(o2m_data[col], errors='coerce')

            # Replace inf and -inf values with NaN
            o2m_data.replace([np.inf, -np.inf], np.nan, inplace=True)

            # Add metadata columns
            o2m_data['user'] = user
            o2m_data['session'] = session

            dataframes.append(o2m_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_O2M = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_O2M_data_0.5HZ.parquet"
        combined_data_O2M.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

### Change frequency ###

In [ ]:
combined_data_O2M = change_frequency(combined_data_O2M, current_hz=0.5, target_hz=4)
combined_data_O2M.to_csv("Biometrics_4HZ/O2M_data_4HZ.csv", index=False)

### LGP ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# Define column names for LGP data
columns = ['LGP']

# List to store file paths
lgp_files = []

# Walk through all subdirectories and collect matching files
for root, dirs, files in os.walk(csv_dir):
    for filename in files:
        if "_LGP_D" in filename or "_LGP_A" in filename:
            lgp_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if lgp_files:
    for file_path in tqdm(lgp_files, desc="Processing LGP files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            lgp_data = pd.read_csv(file_path, header=None, names=columns)
            lgp_data['user'] = user
            lgp_data['session'] = session
            dataframes.append(lgp_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_LGP = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_LGP_data_3HZ.parquet"
        combined_data_LGP.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

### Change Frequency ###

In [ ]:
# Add a time column to represent the current second
combined_data_LGP['time'] = combined_data_LGP.index // 3  # 3 rows

# Create a new dataframe to store 4Hz data
new_data = pd.DataFrame()

# Process each second
for time, group in combined_data_LGP.groupby("time"):
    # Repeat the last row of the group
    repeated_row = group.iloc[-1]
    group_with_repeated = pd.concat([group, repeated_row.to_frame().T], ignore_index=True)
    new_data = pd.concat([new_data, group_with_repeated], ignore_index=True)

# Remove the time column and reset the index
new_data = new_data.drop(columns=["time"]).reset_index(drop=True)

new_data.to_csv("Behavioral_4HZ/LGP_data_4HZ.csv", index=False)


### RGP ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# Define column names for RGP data
columns = ['RGP']

# List to store file paths
rgp_files = []

# Walk through all subdirectories and collect matching files
for root, dirs, files in os.walk(csv_dir):
    for filename in files:
        if "_RGP_D" in filename or "_RGP_A" in filename:
            rgp_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if rgp_files:
    for file_path in tqdm(rgp_files, desc="Processing RGP files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            rgp_data = pd.read_csv(file_path, header=None, names=columns)
            rgp_data['user'] = user
            rgp_data['session'] = session
            dataframes.append(rgp_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_RGP = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_RGP_data_3HZ.parquet"
        combined_data_RGP.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

### Change Frequency ###

In [21]:
# Add a time column to represent the current second
combined_data_RGP['time'] = combined_data_RGP.index // 3  # Each second has 3 rows

# Create a new dataframe to store 4Hz data
new_data = pd.DataFrame()

# Process each second
for time, group in combined_data_RGP.groupby("time"):
    # Repeat the last row of the group
    repeated_row = group.iloc[-1]
    group_with_repeated = pd.concat([group, repeated_row.to_frame().T], ignore_index=True)
    new_data = pd.concat([new_data, group_with_repeated], ignore_index=True)

# Remove the time column and reset the index
new_data = new_data.drop(columns=["time"]).reset_index(drop=True)

new_data.to_csv("Behavioral_4HZ/RGP_data_4HZ.csv", index=False)

### Telemetry ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# List to store file paths
telemetry_files = []

# Walk through all subdirectories and collect matching files
for root, dirs, files in os.walk(csv_dir):
    for filename in files:
        if "_Telemetry_D" in filename or "_Telemetry_A" in filename:
            telemetry_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if telemetry_files:
    for file_path in tqdm(telemetry_files, desc="Processing Telemetry files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            telemetry_data = pd.read_csv(file_path, low_memory=False)
            telemetry_data['user'] = user
            telemetry_data['session'] = session
            dataframes.append(telemetry_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_Telemetry = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_Telemetry_data.parquet"
        combined_data_Telemetry.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

### Fill the missing data with Nan values for user A-A  ###

In [68]:

# Define the number of rows to add
rows_to_add = 144000

# Create a DataFrame with NaN values for 144000 rows
nan_data = pd.DataFrame({
    'user': ['A'] * rows_to_add,
    'session': ['Alert'] * rows_to_add,
    # Add NaN for all other columns that exist in the original DataFrame
    **{col: [float('nan')] * rows_to_add for col in combined_data_Telemetry.columns if col not in ['user', 'session']}
})

# Concatenate the new rows with the existing data
Telemetry_data_updated = pd.concat([nan_data, combined_data_Telemetry], axis=0, ignore_index=True)

# Save the updated DataFrame back to a CSV file (optional)
Telemetry_data_updated.to_parquet("Combined_original_Frequency_Data/combined_Telemetry_data_60HZ.parquet", compression='zstd', index=False)


### Change Frequency ###

In [ ]:
Telemetry_data_updated = change_frequency(Telemetry_data_updated, current_hz=60, target_hz=4)
Telemetry_data_updated.to_csv("Behavioral_4HZ/Telemetry_data_4HZ.csv", index=False)

### FAU ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# List to store file paths
fau_files = []

# Walk through all subdirectories and collect matching files
for root, dirs, files in os.walk(Features_dir):
    for filename in files:
        if "_FAU_D" in filename or "_FAU_A" in filename:
            fau_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if fau_files:
    for file_path in tqdm(fau_files, desc="Processing FAU files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            fau_data = pd.read_csv(file_path, low_memory=False)
            fau_data['user'] = user
            fau_data['session'] = session
            dataframes.append(fau_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_FAU = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_FAU_data_60HZ.parquet"
        combined_data_FAU.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

### Fill missing frames with NaN values ###

In [71]:
# Initialize an empty list to store filled dataframes
filled_dataframes = []

# Process data for each user and session
for (user, session), group in combined_data_FAU.groupby(['user', 'session']):
    # Drop duplicate frames if any exist
    group = group.drop_duplicates(subset='Frame', keep='first')

    # Define the range of frame numbers dynamically
    min_frame = group['Frame'].min()
    max_frame = group['Frame'].max()
    frame_range = np.arange(min_frame, max_frame + 1)

    # Create a complete frame index
    complete_frames = pd.DataFrame({'Frame': frame_range})

    # Merge the complete frame index with the group data
    filled_group = pd.merge(complete_frames, group, on='Frame', how='left')

    # Ensure the total number of rows matches the expected range (cap at MAX_FRAMES if needed)
    filled_group = filled_group[:len(frame_range)]

    # Add user and session information
    filled_group['user'] = user
    filled_group['session'] = session

    # Append to the list
    filled_dataframes.append(filled_group)

# Concatenate all filled dataframes
final_combined_data = pd.concat(filled_dataframes, axis=0, ignore_index=True)

# Save the filled data to a new CSV file
final_combined_data.to_parquet("Combined_original_Frequency_Data/combined_FAU_data_60HZ.parquet",compression='zstd', index=False)


### Change Frequency ###

In [ ]:
combined_data_FAU = change_frequency(final_combined_data, current_hz=60, target_hz=4)
combined_data_FAU.to_csv("Facial_4HZ/FAU_data_4HZ.csv", index=False)

### Pose Landmakrs ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# List to store file paths
pl_files = []

# Walk through all subdirectories and collect matching files
for root, dirs, files in os.walk(Features_dir):
    for filename in files:
        if "_PL_D" in filename or "_PL_A" in filename:
            pl_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if pl_files:
    for file_path in tqdm(pl_files, desc="Processing Pose Landmarks (PL) files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            pl_data = pd.read_csv(file_path, low_memory=False)
            pl_data['user'] = user
            pl_data['session'] = session
            dataframes.append(pl_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_PL = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} Pose Landmark files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_PL_data_60HZ.parquet"
        combined_data_PL.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

### Fill missing frames with NaN values ###

In [74]:
# Initialize an empty list to store filled dataframes
filled_dataframes = []

# Process data for each user and session
for (user, session), group in combined_data_PL.groupby(['user', 'session']):
    # Drop duplicate frames if any exist
    group = group.drop_duplicates(subset='frame', keep='first')

    # Define the range of frame numbers dynamically
    min_frame = group['frame'].min()
    max_frame = group['frame'].max()
    frame_range = np.arange(min_frame, max_frame + 1)

    # Create a complete frame index
    complete_frames = pd.DataFrame({'frame': frame_range})

    # Merge the complete frame index with the group data
    filled_group = pd.merge(complete_frames, group, on='frame', how='left')

    # Ensure the total number of rows matches the expected range (cap at MAX_FRAMES if needed)
    filled_group = filled_group[:len(frame_range)]

    # Add user and session information
    filled_group['user'] = user
    filled_group['session'] = session

    # Append to the list
    filled_dataframes.append(filled_group)

# Concatenate all filled dataframes
final_combined_data = pd.concat(filled_dataframes, axis=0, ignore_index=True)

# Save the filled data to a new CSV file
final_combined_data.to_parquet("Combined_original_Frequency_Data/combined_PL_data_60HZ.parquet",compression='zstd', index=False)


### Change Frequency ###

In [ ]:
combined_data_PL = change_frequency(final_combined_data, current_hz=60, target_hz=4)
combined_data_PL.to_csv("Behavioral_4HZ/PL_data_4HZ.csv", index=False)

### Facial Landmarks ###

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initialize an empty list to store dataframes
dataframes = []

# List to store file paths
fl_files = []

# Walk through all subdirectories and collect matching files
for root, dirs, files in os.walk(Features_dir):
    for filename in files:
        if "_FL_D" in filename or "_FL_A" in filename:
            fl_files.append(os.path.join(root, filename))

# Process files with tqdm progress bar
if fl_files:
    for file_path in tqdm(fl_files, desc="Processing Facial Landmarks (FL) files", unit="file"):
        filename = os.path.basename(file_path)
        parts = filename.split("_")
        user = parts[0]  # First part is the user
        session = "Drowsy" if parts[-1].startswith("D") else "Alert"

        if not os.path.isfile(file_path):
            logging.warning(f"File not found: {file_path}")
            continue

        try:
            fl_data = pd.read_csv(file_path, low_memory=False)
            fl_data['user'] = user
            fl_data['session'] = session
            dataframes.append(fl_data)
        except Exception as e:
            logging.error(f"Error reading {file_path}: {e}")

    # Concatenate all dataframes
    if dataframes:
        combined_data_FL = pd.concat(dataframes, axis=0, ignore_index=True)
        logging.info(f"Successfully combined {len(dataframes)} Facial Landmark files.")

        # Save to Parquet
        output_file = "Combined_original_Frequency_Data/combined_FL_data_60HZ.parquet"
        combined_data_FL.to_parquet(output_file, compression='zstd', index=False)
        logging.info(f"Data saved to {output_file}")
    else:
        logging.warning("No valid data found to process.")
else:
    logging.warning("No matching files found in the directory.")

### Fill missing frames with NaN values ###

In [77]:
# Initialize an empty list to store filled dataframes
filled_dataframes = []

# Process data for each user and session
for (user, session), group in combined_data_FL.groupby(['user', 'session']):
    # Drop duplicate frames if any exist
    group = group.drop_duplicates(subset='Frame_Number', keep='first')

    # Define the range of frame numbers dynamically
    min_frame = group['Frame_Number'].min()
    max_frame = group['Frame_Number'].max()
    frame_range = np.arange(min_frame, max_frame + 1)

    # Create a complete frame index
    complete_frames = pd.DataFrame({'Frame_Number': frame_range})

    # Merge the complete frame index with the group data
    filled_group = pd.merge(complete_frames, group, on='Frame_Number', how='left')

    # Ensure the total number of rows matches the expected range (cap at MAX_FRAMES if needed)
    filled_group = filled_group[:len(frame_range)]

    # Add user and session information
    filled_group['user'] = user
    filled_group['session'] = session

    # Append to the list
    filled_dataframes.append(filled_group)

# Concatenate all filled dataframes
final_combined_data_fl = pd.concat(filled_dataframes, axis=0, ignore_index=True)

# Save the filled data to a new CSV file
final_combined_data_fl.to_parquet("Combined_original_Frequency_Data/combined_FL_data_60HZ.parquet",compression='zstd', index=False)


### Change Frequency ###

In [ ]:
combined_data_FL = change_frequency(final_combined_data_fl, current_hz=60, target_hz=4)
combined_data_FL.to_csv("Facial_4HZ/FL_data_4HZ.csv", index=False)